# Part B - Extraction  (Notebook 2 of 2)

Consumes the **masked** drawings produced by `PartA_Masking.ipynb` and extracts a
structured fastener record from each sheet.

Pipeline: **GPT extractor || Claude extractor** (in parallel) -> **GPT arbiter**
reconciles them pixel-by-pixel -> color-coded **XLSX** + per-drawing audit JSON.

**Inputs**
- Masked images: either already present in `masked_drawings/` (same Colab runtime
  as Part A) **or** uploaded as `masked_drawings.zip` when prompted below.
- API keys: read from **`secret_keys.env`** (falls back to OS env vars / Colab Secrets).

**Confidentiality:** because Part A already blacked out the company name + logo,
the images sent to the OpenAI / Anthropic APIs here contain no owner branding.

### B-install - Extraction dependencies

In [ ]:
# Extraction stack only (cloud APIs + image/spreadsheet IO). No local VLM/GPU.
!apt-get -qq install -y poppler-utils          # for pdf2image
!pip install -q openai anthropic pdf2image pillow openpyxl pandas
print("Extraction dependencies installed.")

### B0 - API keys + clients  (loaded from secret_keys.env)

In [ ]:
import os

# Where to look for the shared secrets file (first hit wins).
from pathlib import Path as _P
def _root():
    for d in [_P.cwd(), *_P.cwd().resolve().parents]:
        if (d / "src").is_dir() and (d / "notebooks").is_dir():
            return d
    return _P.cwd()
SECRETS_FILE_CANDIDATES = [str(_P.cwd() / "secret_keys.env"),
                           str(_root() / "secret_keys.env"),
                           "/content/secret_keys.env"]

def _parse_env_file(path):
    out = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            out[k.strip()] = v.strip().strip('"').strip("'")
    return out

def load_keys(names=("OPENAI_API_KEY", "ANTHROPIC_API_KEY")):
    found = {}
    # 1) secrets file
    for p in SECRETS_FILE_CANDIDATES:
        if os.path.isfile(p):
            try:
                fk = _parse_env_file(p)
                for n in names:
                    if fk.get(n) and not fk[n].endswith("REPLACE_ME"):
                        found.setdefault(n, fk[n])
                print("Loaded keys from", p)
                break
            except Exception as e:
                print("Could not parse", p, ":", e)
    # 2) OS environment
    for n in names:
        if not found.get(n) and os.environ.get(n):
            found[n] = os.environ[n]
    # 3) Colab Secrets panel
    try:
        from google.colab import userdata
        for n in names:
            if not found.get(n):
                try:
                    v = userdata.get(n)
                    if v:
                        found[n] = v
                except Exception:
                    pass
    except Exception:
        pass
    # export so any downstream library can read them too
    for n, v in found.items():
        os.environ[n] = v
    missing = [n for n in names if not found.get(n)]
    if missing:
        print("WARNING - missing keys:", missing,
              "-> edit secret_keys.env (or set env vars / Colab Secrets).")
    return found

KEYS = load_keys()

from openai import OpenAI
from anthropic import Anthropic
client           = OpenAI(api_key=KEYS.get("OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=KEYS.get("ANTHROPIC_API_KEY"))
print("OpenAI + Anthropic clients ready.")

### B-input - Get the masked images from Part A

In [ ]:
import os, glob, shutil

# Shared hand-off convention with Part A.
from pathlib import Path

def _on_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

def _repo_root():
    for d in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (d / "src").is_dir() and (d / "notebooks").is_dir():
            return d
    return Path.cwd()

def _handoff_dir():
    # masking OUTPUT == extraction INPUT (identical on both notebooks)
    d = Path("/content/masked_drawings") if _on_colab() \
        else _repo_root() / "data" / "interim" / "masked_drawings"
    d.mkdir(parents=True, exist_ok=True)
    return d

# Shared hand-off convention with Part A (identical folder).
MASKED_INPUT_DIR = str(_handoff_dir())
os.makedirs(MASKED_INPUT_DIR, exist_ok=True)

def _masked_present():
    return [p for p in glob.glob(f"{MASKED_INPUT_DIR}/*")
            if p.lower().endswith((".png", ".jpg", ".jpeg", ".pdf"))]

existing = _masked_present()
if existing:
    print(f"Found {len(existing)} masked image(s) already in {MASKED_INPUT_DIR} "
          f"(same runtime as Part A).")
else:
    print("No masked images found. Upload the hand-off 'masked_drawings.zip' from Part A...")
    try:
        from google.colab import files as colab_files
        import zipfile, io
        up = colab_files.upload()                      # pick masked_drawings.zip (or loose images)
        for name, data in up.items():
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(data)) as z:
                    z.extractall(MASKED_INPUT_DIR)
            else:
                with open(os.path.join(MASKED_INPUT_DIR, name), "wb") as fh:
                    fh.write(data)
    except Exception as e:
        print("Not on Colab - manually place masked images into",
              MASKED_INPUT_DIR, ":", e)
    # flatten any subfolder the zip may have created
    for p in glob.glob(f"{MASKED_INPUT_DIR}/**/*", recursive=True):
        if p.lower().endswith((".png", ".jpg", ".jpeg", ".pdf")) and \
           os.path.dirname(p) != MASKED_INPUT_DIR:
            shutil.move(p, os.path.join(MASKED_INPUT_DIR, os.path.basename(p)))

final = _masked_present()
print(f"{len(final)} masked file(s) ready in {MASKED_INPUT_DIR}:",
      [os.path.basename(p) for p in final][:10])

### B1 · Config — input is `MASKED_INPUT_DIR`

In [ ]:
import os

# Parallel extractors
EXTRACTOR_GPT     = "gpt-5.4"
EXTRACTOR_CLAUDE  = "claude-opus-4-7"
# Arbiter
ARBITER_MODEL     = "gpt-5.5"

# Reasoning effort
REASONING_EXTRACTOR = "medium"
REASONING_ARBITER   = "high"

IMAGE_DETAIL      = "high"
MAX_EDGE_PX       = 2048
PDF_DPI           = 200
CLAUDE_MAX_TOKENS = 4096
STATELESS_CALLS   = True

DRAWING_DIR = MASKED_INPUT_DIR
OUT_XLSX    = "/content/parallel_arbiter_results.xlsx"
JSON_DIR    = "/content/audit_json"
os.makedirs(JSON_DIR, exist_ok=True)

print(f"DRAWING_DIR = {DRAWING_DIR} | exists: {os.path.isdir(DRAWING_DIR)}")
if os.path.isdir(DRAWING_DIR):
    contents = os.listdir(DRAWING_DIR)
    print(f"  {len(contents)} item(s):", contents[:10])
    if not contents:
        print("  ⚠️  Empty — upload via the Files panel (📁 left), then re-run.")

### B2 · Schema v4

In [ ]:
def s(desc): return {"type":"string","description":desc}
def s_enum(vals, desc): return {"type":"string","enum":vals,"description":desc}

GDT_SYMBOLS = ["straightness","flatness","circularity","cylindricity",
    "profile_of_a_line","profile_of_a_surface","perpendicularity","parallelism",
    "angularity","position","concentricity","symmetry","circular_runout","total_runout","NULL"]

EXTRACTION_SCHEMA = {
  "type":"object","additionalProperties":False,
  "properties":{
    "drawing_number": s("Part/drawing number from title block, or NULL"),
    "revision":       s("Revision letter/number, or NULL"),
    "drawing_type":   s_enum(["single_part_drawing","catalog_sheet","assembly_drawing"], "Type"),
    "part_family":    s_enum(["hex_bolt","shcs","button_head","csk","pan_head","hex_nut",
                              "flanged_nut","washer","pin","stud","rivet","bush","custom_special"], "Family"),
    "category":       s("Functional part name as written, or NULL"),
    "standard_reference": s("PART standard e.g. ISO 4014, DIN 912. NOT material standard. NULL if none."),
    "stated_mass_g":  s("Mass in grams if printed (convert kg→g), or NULL"),
    "material_family": s_enum(["carbon_steel","alloy_steel","stainless_steel","brass","aluminium",
                               "titanium","plastic","NULL"], "Base material"),
    "material_grade": s("Grade/property class e.g. 10.9, A2-70, EN8, or NULL"),
    "material_standard_ref": s("MATERIAL standard e.g. EN 10263-4, ASTM A574, or NULL"),
    "material_condition": s("Q+T, cold drawn, annealed, or NULL"),
    "material_verbatim": s("Exact raw material string verbatim, or NULL"),
    "thread_designation": s("e.g. M10x1.5-6g, or NULL"),
    "nominal_diameter_mm": s("Nominal diameter mm number-string, or NULL"),
    "thread_pitch_mm": s("Thread pitch mm. NULL only if not printed AND not recoverable"),
    "nominal_length_mm": s("Nominal length mm number-string, or NULL"),
    "dimensions": {"type":"array","minItems":1,
      "description":"All family-specific dimensions as {feature_name, value, unit}. VALUE is the NUMBER. Never empty.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{
          "feature_name": s("Canonical name+symbol e.g. head_across_flats_s — OR '__extraction_failure__'"),
          "value":        s("The NUMBER, e.g. '16'. 'NULL' literal only inside a failure entry."),
          "unit":         s_enum(["mm","deg","NULL"], "Unit")},
        "required":["feature_name","value","unit"]}},
    "dimension_table_verbatim": s("Raw text of the dim table incl. header and selected-row marker. NULL if no table."),
    "hardness_value": s("Hardness number or range, or NULL"),
    "hardness_scale": s_enum(["HRC","HRB","HV","HB","NULL"], "Scale — never guess"),
    "tensile_strength_mpa": s("Tensile MPa only if EXPLICITLY stated, or NULL"),
    "yield_strength_mpa": s("Yield MPa only if EXPLICITLY stated, or NULL"),
    "heat_treatment": s("e.g. Q+T to HRC 32-39, or NULL"),
    "surface_treatment": s("Coating/plating. DOUBLE-CHECK by rescanning the sheet."),
    "surface_treatment_thickness_um": s("Thickness µm, or NULL"),
    "surface_treatment_standard": s("e.g. ISO 4042, or NULL"),
    "surface_roughness_ra_um": s("Ra µm (√ symbol) e.g. 3.2, or NULL"),
    "general_tolerance_class": s("e.g. ISO 2768-m, or NULL"),
    "has_gdt": s_enum(["true","false"], "true only if a real feature control frame exists"),
    "gdt_callouts": {"type":"array","description":"GD&T frames. [] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"symbol": s_enum(GDT_SYMBOLS,"characteristic"),
                      "tolerance_value": s("e.g. 0.05"),
                      "primary_datum":   s("or NULL"),
                      "secondary_datum": s("or NULL"),
                      "tertiary_datum":  s("or NULL")},
        "required":["symbol","tolerance_value","primary_datum","secondary_datum","tertiary_datum"]}},
    "critical_tolerances": {"type":"array","description":"[] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"feature": s("e.g. shank diameter"), "tolerance": s("e.g. 10 h7")},
        "required":["feature","tolerance"]}},
    "feature_count": {"type":"array",
      "description":"Counts literally stated, e.g. 'No. of knurlings: 4'. [] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"feature": s("e.g. knurlings, turns, holes"),
                      "count":   s("Integer as string, exactly as written")},
        "required":["feature","count"]}},
    "marking_requirements": s("Head stamping / lot marking, or NULL"),
    "inspection_requirements": s("e.g. 100% MPI, AQL 1.0, or NULL"),
    "packaging_requirements": s("e.g. 500 pcs/box bulk, or NULL"),
    "manufacture_process_instruction": s("Process instructions, esp. conditional, verbatim. NULL if none."),
    "other_notes": s("Other instructions not in the above buckets. NULL if none."),
    "notes_verbatim": s("FULL consolidated raw text of EVERY text region. NULL only if none."),
    "low_confidence_fields": s("Comma-separated uncertain field names, or NULL"),
    "self_reported_confidence": s("Overall confidence 0.0-1.0 number-string"),
  },
  "required":["drawing_number","revision","drawing_type","part_family","category","standard_reference",
    "stated_mass_g","material_family","material_grade","material_standard_ref","material_condition",
    "material_verbatim","thread_designation","nominal_diameter_mm","thread_pitch_mm","nominal_length_mm",
    "dimensions","dimension_table_verbatim","hardness_value","hardness_scale","tensile_strength_mpa",
    "yield_strength_mpa","heat_treatment","surface_treatment","surface_treatment_thickness_um",
    "surface_treatment_standard","surface_roughness_ra_um","general_tolerance_class","has_gdt","gdt_callouts",
    "critical_tolerances","feature_count","marking_requirements","inspection_requirements",
    "packaging_requirements","manufacture_process_instruction","other_notes","notes_verbatim",
    "low_confidence_fields","self_reported_confidence"],
}

ARRAY_FIELDS = {"dimensions":"dimensions_json","gdt_callouts":"gdt_callouts_json",
                "critical_tolerances":"critical_tolerances_json","feature_count":"feature_count_json"}
COLUMN_ORDER = ["filename","source_file","source_page"] + [ARRAY_FIELDS.get(f, f) for f in EXTRACTION_SCHEMA["required"]]
DATA_COLS    = [ARRAY_FIELDS.get(f, f) for f in EXTRACTION_SCHEMA["required"]]
print(f"✅ Schema v4: {len(EXTRACTION_SCHEMA['required'])} fields | total cols: {len(COLUMN_ORDER)}")

### B3 · Prompts

In [ ]:

EXTRACTOR_SYSTEM = """You are a senior mechanical engineer specialising in fastener design, GD&T, and engineering-drawing interpretation per ISO, DIN, EN, ASME and JIS standards.

ANTI-CARRYOVER: Treat THIS drawing as your only context. Re-read every region of THIS sheet before producing any field. Do not reuse values by analogy.

CORE RULES:
- Extract only what is shown. Sweep the ENTIRE sheet, including HIGHLIGHTED / boxed / shaded / circled cells and every table.
- Multiple text regions may exist (title block, corner notes, notes near views, flag notes, revision table, bordered tables). notes_verbatim concatenates them all.
- Exact terminology: top-view hex head = 'width across flats' (s); vertical head dim = 'head height' (k); socket recess = 'hex socket across flats'.
- Distinguish surface ROUGHNESS (Ra µm, √) from surface TREATMENT (plating/coating).
- Distinguish PART standard (standard_reference) from MATERIAL standard.
- Hardness scales HRC/HRB/HV/HB. Number with no scale → hardness_scale=NULL. Never guess.

DIMENSION RESOLUTION (internal — do NOT expose in output):
A. Printed at feature → read it.
B. Letter callout (L, D, A...) → find table, find COLOUR-HIGHLIGHTED / boxed / circled selected row, read the number. Copy the table into dimension_table_verbatim.
C. If A/B fail and a standard is referenced → resolve from standard.
D. Otherwise skip the feature — never invent.

DIMENSION OUTPUT RULES:
- Each item is exactly {feature_name, value, unit}. VALUE is the NUMBER, never a letter.
- 'dimensions' is never empty. If nothing recoverable → ONE entry {feature_name='__extraction_failure__', value='NULL', unit='NULL'}.

NON-DIMENSION fields: extract only what is shown. No standard-based inference.
NOTES SPLIT: conditional manufacturing instructions → manufacture_process_instruction verbatim.
feature_count: only when literally stated. [] otherwise.

STRICT NULL POLICY: scalar absent → "NULL"; arrays use []; dimensions uses failure entry."""

EXTRACTOR_USER = """Single engineering drawing of a fastener/part. This drawing only — do not reference any other.

Fill every schema field. Promote nominal_diameter/pitch/length when explicit or table-resolvable. All other dims in 'dimensions' as {feature_name,value,unit} with NUMBER values. Copy any dimension table into dimension_table_verbatim. Apply strict NULL/empty-array policy."""


# ── Arbiter (GPT-5.5) — sees image + both sheets; SOFT preference for Claude on logical conflict ──
ARBITER_SYSTEM = """You are GPT-5.5, the final arbiter on a fastener-drawing extraction. Two independent vision models have each produced their own extraction of THIS drawing:

  • Sheet-1 — produced by GPT-5.4 (an earlier model in YOUR OWN family).
  • Sheet-2 — produced by Claude (a different model family).

You also see the original drawing image. Your job is to produce the FINAL reconciled extraction by analysing the image pixel-by-pixel and weighing both candidates.

BIAS-AWARENESS DISCIPLINE (read carefully):
- Sheet-1 comes from your own model family. There is a known risk that you would over-trust it because it "sounds like you". You must actively guard against this.
- Therefore, on LOGICAL/SUBSTANTIVE conflicts (where the two candidates disagree on what the drawing actually says — e.g. different numbers, different materials, different categories — NOT trivial wording differences), apply this rule:
    1. Re-examine the relevant region of the image yourself, carefully.
    2. If your independent reading of the pixels CLEARLY matches one candidate, pick that one regardless of which source it is.
    3. If both candidates are equally defensible by the pixels — i.e. you genuinely cannot tell which is right, then if it is a value not equal to NULL given by Claude — DEFAULT TO CLAUDE'S VALUE (Sheet-2). This is the soft tiebreaker to neutralise same-family bias. Mark chosen_from='claude'.
    4. If neither candidate matches what you see in the image, produce the value you actually see and mark chosen_from='synthesized'.
    5. In case where one model says a given value, whereas other model says NULL, then revise that particular part of the image twice on your own and take your own independent decision; if you cannot conclude, then pick the value that is not NULL.
- On MINOR WORDING differences (synonyms, casing, equivalent normalisations), do not invoke the tiebreaker — pick whichever is most canonical and mark chosen_from='consensus'.

DECISION RULES per field:
- Both agree → use that value; agreement '2/2'; confidence 0.9–1.0; chosen_from='consensus'.
- One is NULL, the other has a value the image supports → use the value; '1/2'; confidence 0.7–0.85.
- Logical conflict resolved by image → the value the image supports; '1/2'; confidence 0.6–0.85.
- Logical conflict, image inconclusive → CLAUDE'S value (soft tiebreaker); '1/2'; confidence 0.4–0.6.
- Neither matches the image → your synthesized value; '0/2'; confidence 0.3–0.55; chosen_from='synthesized'.
- Both NULL → keep NULL; '2/2'; confidence 1.0.

DIMENSIONS — CRITICAL:
- Each dimension is exactly {feature_name, value, unit}. VALUE must be a NUMBER (never a letter L/D/A).
- If either candidate has a letter as a value, resolve it from dimension_table_verbatim (highlighted row) or the image, and emit the number.
- 'dimensions' is never empty — failure entry if nothing recoverable.

TAKE YOUR TIME — examine every field deliberately. This is the canonical output."""

ARBITER_USER_TEMPLATE = """The drawing image is attached. Below are two independent extractions of it.

Sheet-1 (GPT-5.4 — same family as you; be bias-aware):
{sheet1_json}

Sheet-2 (Claude — different family; soft tiebreaker preference on logical conflicts only):
{sheet2_json}

Re-read the drawing pixel-by-pixel. For every field, produce the final value plus meta (confidence, agreement '2/2'|'1/2'|'0/2', chosen_from 'gpt-5.4'|'claude'|'consensus'|'synthesized'). Return final + meta strictly per the schema."""

print("✅ Prompts ready: extractors (Sheets 1+2) + arbiter (Sheet 3 with soft Claude tiebreaker).")

### B4 · Helpers

In [ ]:
import base64, io, json
from PIL import Image
from pdf2image import convert_from_path

def load_pages(path):
    if path.lower().endswith(".pdf"):
        return convert_from_path(path, dpi=PDF_DPI)
    return [Image.open(path).convert("RGB")]

def img_b64(pil_img):
    img = pil_img.convert("RGB")
    w, h = img.size; lo = max(w, h)
    if lo > MAX_EDGE_PX:
        sc = MAX_EDGE_PX/lo
        img = img.resize((round(w*sc), round(h*sc)), Image.LANCZOS)
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def coerce(v):
    bad = {"","n/a","na","none","null","-","—"}
    if v is None: return "NULL"
    if isinstance(v, str): return "NULL" if v.strip().lower() in bad else v.strip()
    return str(v)

def flatten(raw: dict) -> dict:
    row = {}
    for f in EXTRACTION_SCHEMA["required"]:
        if f in ARRAY_FIELDS:
            v = raw.get(f, [])
            row[ARRAY_FIELDS[f]] = json.dumps(v, ensure_ascii=False) if v else "NULL"
        else:
            row[f] = coerce(raw.get(f))
    return row

def unflatten_for_prompt(row: dict) -> dict:
    out = {}
    for f in EXTRACTION_SCHEMA["required"]:
        if f in ARRAY_FIELDS:
            v = row.get(ARRAY_FIELDS[f], "NULL")
            try: out[f] = json.loads(v) if v != "NULL" else []
            except Exception: out[f] = []
        else:
            out[f] = row.get(f, "NULL")
    return out

def per_drawing_preamble(filename, source_file, page_idx):
    return (f"NEW DRAWING — extract this sheet only.\n"
            f"filename: {filename} | source: {source_file} | page: {page_idx}\n"
            f"Treat any prior conversation or context as irrelevant. Read the pixels.\n")

print("✅ Helpers ready.")

### B5 · Parallel extractors

In [ ]:
def gpt_extract(pil_img, preamble):
    content = [{"type":"input_text","text": preamble + "\n" + EXTRACTOR_USER},
               {"type":"input_image",
                "image_url":f"data:image/png;base64,{img_b64(pil_img)}","detail":IMAGE_DETAIL}]
    r = client.responses.create(
        model=EXTRACTOR_GPT, reasoning={"effort":REASONING_EXTRACTOR},
        store=not STATELESS_CALLS,
        input=[{"role":"system","content":[{"type":"input_text","text":EXTRACTOR_SYSTEM}]},
               {"role":"user","content":content}],
        text={"format":{"type":"json_schema","name":"fastener_extraction",
                        "strict":True,"schema":EXTRACTION_SCHEMA}})
    return flatten(json.loads(r.output_text))

_CLAUDE_TOOL = {"name":"emit_extraction",
                "description":"Return the fastener extraction record exactly per schema.",
                "input_schema":EXTRACTION_SCHEMA}
def claude_extract(pil_img, preamble):
    blocks = [{"type":"image","source":{"type":"base64","media_type":"image/png","data":img_b64(pil_img)}},
              {"type":"text","text": preamble + "\n" + EXTRACTOR_USER}]
    resp = anthropic_client.messages.create(
        model=EXTRACTOR_CLAUDE, max_tokens=CLAUDE_MAX_TOKENS, system=EXTRACTOR_SYSTEM,
        tools=[_CLAUDE_TOOL], tool_choice={"type":"tool","name":"emit_extraction"},
        messages=[{"role":"user","content":blocks}])
    tool_block = next(b for b in resp.content if b.type == "tool_use")
    return flatten(tool_block.input)

print("✅ Cell 7: parallel extractors ready (5.4 ‖ Claude).")

### B6 · Arbiter

In [ ]:
def build_arbiter_schema(data_cols):
    fp = {c: {"type":"string"} for c in data_cols}
    mp = {c: {"type":"object","additionalProperties":False,
        "properties":{"confidence":{"type":"string","description":"0.0-1.0"},
            "agreement":{"type":"string","enum":["2/2","1/2","0/2"]},
            "chosen_from":{"type":"string","enum":["gpt-5.4","claude","consensus","synthesized"]}},
        "required":["confidence","agreement","chosen_from"]} for c in data_cols}
    return {"type":"object","additionalProperties":False,"properties":{
        "final":{"type":"object","additionalProperties":False,"properties":fp,"required":list(data_cols)},
        "meta": {"type":"object","additionalProperties":False,"properties":mp,"required":list(data_cols)}},
        "required":["final","meta"]}
ARBITER_SCHEMA = build_arbiter_schema(DATA_COLS)

def arbiter(pil_img, sheet1_row, sheet2_row, preamble):
    s1 = unflatten_for_prompt(sheet1_row)
    s2 = unflatten_for_prompt(sheet2_row)
    user_text = ARBITER_USER_TEMPLATE.format(
        sheet1_json=json.dumps(s1, ensure_ascii=False, indent=2),
        sheet2_json=json.dumps(s2, ensure_ascii=False, indent=2))
    content = [{"type":"input_text","text": preamble + "\n" + user_text},
               {"type":"input_image",
                "image_url":f"data:image/png;base64,{img_b64(pil_img)}","detail":IMAGE_DETAIL}]
    r = client.responses.create(
        model=ARBITER_MODEL, reasoning={"effort":REASONING_ARBITER},
        store=not STATELESS_CALLS,
        input=[{"role":"system","content":[{"type":"input_text","text":ARBITER_SYSTEM}]},
               {"role":"user","content":content}],
        text={"format":{"type":"json_schema","name":"arbiter_reconciliation",
                        "strict":True,"schema":ARBITER_SCHEMA}})
    out = json.loads(r.output_text)
    return out["final"], out["meta"]

print("✅ Cell 8: arbiter ready (GPT-5.5 high-reasoning, soft Claude tiebreaker).")

### B7 · Driver

In [ ]:
from pathlib import Path

files = sorted([str(p) for p in Path(DRAWING_DIR).iterdir()
                if p.suffix.lower() in (".pdf",".jpg",".jpeg",".png")])
print(f"{len(files)} files → 5.4 ‖ Claude → 5.5 arbiter\n")

rows_s1, rows_s2, rows_s3 = [], [], []
rows_conf = []

import re
_PAGE_RE = re.compile(r"_p(\d+)$")

for i, fp in enumerate(files, 1):
    src = os.path.basename(fp)
    for pidx, page in enumerate(load_pages(fp), 1):
        stem = Path(src).stem
        m = _PAGE_RE.search(stem)
        page_from_name = int(m.group(1)) if m else pidx
        fn = stem if m else (f"{stem}_p{pidx}" if fp.lower().endswith(".pdf") else stem)
        preamble = per_drawing_preamble(fn, src, page_from_name)
        print(f"[{i}/{len(files)}] {fn} ...", end=" ")
        try:
            ...
            # Sheet-1 & Sheet-2 INDEPENDENT (no shared state)
            try: row1 = gpt_extract(page, preamble); err1 = ""
            except Exception as e: row1 = {c:"NULL" for c in DATA_COLS}; err1 = str(e)[:200]
            try: row2 = claude_extract(page, preamble); err2 = ""
            except Exception as e: row2 = {c:"NULL" for c in DATA_COLS}; err2 = str(e)[:200]

            # Sheet-3: arbiter — sees both + image
            try:
                final, meta = arbiter(page, row1, row2, preamble)
                row3 = {c: coerce(final.get(c)) for c in DATA_COLS}
                conf = {c: float(meta.get(c,{}).get("confidence","0") or 0) for c in DATA_COLS}
                chosen = {c: meta.get(c,{}).get("chosen_from","") for c in DATA_COLS}
            except Exception as e:
                # arbiter failure → fall back to whichever extractor succeeded
                fb = row2 if not err2 else (row1 if not err1 else {c:"NULL" for c in DATA_COLS})
                row3 = dict(fb); conf = {c: 0.0 for c in DATA_COLS}; chosen = {c: "fallback" for c in DATA_COLS}

            mean_conf = round(sum(conf.values())/len(conf), 3) if conf else 0.0

            # tag metadata + three-source confidence on Sheet-3
            for r in (row1, row2, row3):
                r["filename"]=fn; r["source_file"]=src; r["source_page"]=pidx
            row3["conf_gpt54_self"]   = row1.get("self_reported_confidence", "NULL")
            row3["conf_claude_self"]  = row2.get("self_reported_confidence", "NULL")
            row3["conf_gpt55_judged"] = str(mean_conf)
            row3["models_ok"]         = ",".join([t for t,e in [("gpt-5.4",err1),("claude",err2)] if not e])

            # confidence row (same shape, for the colouring lookup)
            cr = {c: conf.get(c, 0.0) for c in DATA_COLS}
            cr["filename"]=fn; cr["source_file"]=src; cr["source_page"]=pidx
            cr["conf_gpt54_self"]=row3["conf_gpt54_self"]; cr["conf_claude_self"]=row3["conf_claude_self"]
            cr["conf_gpt55_judged"]=mean_conf

            rows_s1.append(row1); rows_s2.append(row2); rows_s3.append(row3); rows_conf.append(cr)

            with open(os.path.join(JSON_DIR, f"{fn}_pipeline.json"), "w") as f:
                json.dump({"sheet1":row1,"sheet2":row2,"sheet3":row3,
                           "meta":meta if 'meta' in locals() else {},"chosen_from":chosen,
                           "errors":{"sheet1":err1,"sheet2":err2}}, f, indent=2, ensure_ascii=False)
            print(f"✅ ok={row3['models_ok']}  judged_conf={mean_conf}")
        except Exception as e:
            stub = {c:"NULL" for c in COLUMN_ORDER}
            stub["filename"]=fn; stub["source_file"]=src; stub["source_page"]=pidx
            rows_s1.append(dict(stub)); rows_s2.append(dict(stub)); rows_s3.append(dict(stub))
            cr = {c: 0.0 for c in DATA_COLS}; cr["filename"]=fn; cr["source_file"]=src; cr["source_page"]=pidx
            rows_conf.append(cr)
            print(f"❌ {str(e)[:120]}")

### B8 · Write XLSX

In [ ]:
import pandas as pd
from openpyxl.styles import PatternFill

SHEET3_EXTRA = ["conf_gpt54_self","conf_claude_self","conf_gpt55_judged","models_ok"]
FULL_S12  = COLUMN_ORDER
FULL_S3   = COLUMN_ORDER + SHEET3_EXTRA
FULL_CONF = COLUMN_ORDER + ["conf_gpt54_self","conf_claude_self","conf_gpt55_judged"]

def frame(rows, cols, fill="NULL"):
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns: df[c] = fill
    return df[cols]

df_s1   = frame(rows_s1,   FULL_S12).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_s2   = frame(rows_s2,   FULL_S12).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_s3   = frame(rows_s3,   FULL_S3 ).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_conf = frame(rows_conf, FULL_CONF, fill=0.0)

GREEN=PatternFill("solid",fgColor="C6EFCE"); AMBER=PatternFill("solid",fgColor="FFEB9C"); RED=PatternFill("solid",fgColor="FFC7CE")
def fill_for(c):
    try: c=float(c)
    except: return None
    return GREEN if c>=0.85 else AMBER if c>=0.5 else RED
meta_cols = {"filename","source_file","source_page","models_ok",
             "conf_gpt54_self","conf_claude_self","conf_gpt55_judged"}

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df_s3.to_excel(w, sheet_name="Sheet3_FINAL_arbiter", index=False)   # canonical first
    df_s1.to_excel(w, sheet_name="Sheet1_GPT54",        index=False)
    df_s2.to_excel(w, sheet_name="Sheet2_Claude",        index=False)
    df_conf.to_excel(w, sheet_name="confidence_per_cell", index=False)
    # colour Sheet3 by per-cell judged confidence
    ws = w.sheets["Sheet3_FINAL_arbiter"]; cols=list(df_s3.columns)
    for r in range(len(df_s3)):
        for ci, col in enumerate(cols):
            if col in meta_cols: continue
            f = fill_for(df_conf.iloc[r].get(col, 0.0))
            if f: ws.cell(row=r+2, column=ci+1).fill = f

print(f"✅ XLSX → {OUT_XLSX}")
print(f"   • Sheet3_FINAL_arbiter   ← canonical (color-coded by per-cell confidence)")
print(f"   • Sheet1_GPT54           ← raw output from GPT-5.4")
print(f"   • Sheet2_Claude          ← raw output from Claude")
print(f"   • confidence_per_cell    ← per-cell judged confidence values")
print(f"✅ per-drawing JSON → {JSON_DIR}/*_pipeline.json\n")

# console summary
show=[c for c in df_s3.columns if c not in meta_cols]
for i in range(len(df_s3)):
    print(f"\n{'='*84}\n {df_s3.iloc[i]['filename']} | models_ok={df_s3.iloc[i]['models_ok']} | "
          f"5.4_self={df_s3.iloc[i]['conf_gpt54_self']} claude_self={df_s3.iloc[i]['conf_claude_self']} "
          f"5.5_judged={df_s3.iloc[i]['conf_gpt55_judged']}\n{'='*84}")
    comp = pd.DataFrame({
        "field":   show,
        "5.4":     [str(df_s1.iloc[i][c])[:24] if c in df_s1.columns else "" for c in show],
        "claude":  [str(df_s2.iloc[i][c])[:24] if c in df_s2.columns else "" for c in show],
        "FINAL":   [str(df_s3.iloc[i][c])[:24] for c in show],
        "conf":    [df_conf.iloc[i][c]        for c in show],
    })
    print(comp.to_string(index=False))

## 4 · Download results

In [ ]:
from google.colab import files as colab_files
colab_files.download(OUT_XLSX)